In [1]:
import pandas as pd
import numpy as np
import itertools
import datetime
import pandas_gbq
import matplotlib.pyplot as plt
from datetime import *
from datetime import datetime, timedelta, date
from pathlib import Path
from PIL import Image
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
project_id = "perceptive-ivy-290216"

# Standard plotly imports
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
query4=f"""
SELECT * FROM `perceptive-ivy-290216.f1_api.results_qualifying`
# WHERE GP="Austrian Grand Prix"
# AND YEAR=2005
"""
track_qualy_all=pandas_gbq.read_gbq(query4,project_id,dialect='standard')

Downloading: 100%|██████████|


In [3]:
track_qualy_all.head()

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Year,GP
0,61,J DOOHAN,DOO,doohan,Alpine,0093cc,alpine,Jack,Doohan,Jack Doohan,None,AUS,20.0,,NaN,0 days 00:01:24.105000,NaT,NaT,NaT,,NaN,2024,Abu Dhabi Grand Prix
1,10,P GASLY,GAS,nan,Alpine,0093cc,nan,Pierre,Gasly,Pierre Gasly,https://media.formula1.com/d_driver_fallback_i...,FRA,NaN,,NaN,NaT,NaT,NaT,NaT,,NaN,2024,Azerbaijan Grand Prix
2,10,P GASLY,GAS,gasly,Alpine,0093cc,alpine,Pierre,Gasly,Pierre Gasly,https://media.formula1.com/d_driver_fallback_i...,FRA,18.0,,NaN,0 days 00:01:31.312000,NaT,NaT,NaT,,NaN,2024,Singapore Grand Prix
3,10,P GASLY,GAS,gasly,Alpine,0093cc,alpine,Pierre,Gasly,Pierre Gasly,https://media.formula1.com/d_driver_fallback_i...,FRA,7.0,,NaN,0 days 00:01:33.550000,0 days 00:01:33.162000,0 days 00:01:33.018000,NaT,,NaN,2024,United States Grand Prix
4,10,P GASLY,GAS,gasly,Alpine,0093cc,alpine,Pierre,Gasly,Pierre Gasly,https://media.formula1.com/d_driver_fallback_i...,FRA,8.0,,NaN,0 days 00:01:17.149000,0 days 00:01:17.048000,0 days 00:01:16.892000,NaT,,NaN,2024,Mexico City Grand Prix


In [4]:
track4=pd.melt(track_qualy_all, 
                  id_vars=['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName',
       'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName','HeadshotUrl', 'CountryCode', 'Position', 
       'ClassifiedPosition','GridPosition', 'Time', 'Status', 'Points', 'Year','GP'],  # Columns to keep as identifiers
        value_vars=['Q1', 'Q2', 'Q3'], # Columns to unpivot
        var_name='Type', # Name for the new column containing variable names
        value_name='LapTime') # Name for the new column containing values
track4.head()

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,Position,ClassifiedPosition,GridPosition,Time,Status,Points,Year,GP,Type,LapTime
0,61,J DOOHAN,DOO,doohan,Alpine,0093cc,alpine,Jack,Doohan,Jack Doohan,None,AUS,20.0,,NaN,NaT,,NaN,2024,Abu Dhabi Grand Prix,Q1,0 days 00:01:24.105000
1,10,P GASLY,GAS,nan,Alpine,0093cc,nan,Pierre,Gasly,Pierre Gasly,https://media.formula1.com/d_driver_fallback_i...,FRA,NaN,,NaN,NaT,,NaN,2024,Azerbaijan Grand Prix,Q1,NaT
2,10,P GASLY,GAS,gasly,Alpine,0093cc,alpine,Pierre,Gasly,Pierre Gasly,https://media.formula1.com/d_driver_fallback_i...,FRA,18.0,,NaN,NaT,,NaN,2024,Singapore Grand Prix,Q1,0 days 00:01:31.312000
3,10,P GASLY,GAS,gasly,Alpine,0093cc,alpine,Pierre,Gasly,Pierre Gasly,https://media.formula1.com/d_driver_fallback_i...,FRA,7.0,,NaN,NaT,,NaN,2024,United States Grand Prix,Q1,0 days 00:01:33.550000
4,10,P GASLY,GAS,gasly,Alpine,0093cc,alpine,Pierre,Gasly,Pierre Gasly,https://media.formula1.com/d_driver_fallback_i...,FRA,8.0,,NaN,NaT,,NaN,2024,Mexico City Grand Prix,Q1,0 days 00:01:17.149000


In [5]:
track4['LapTime_TD']= pd.to_timedelta(track4["LapTime"])
track4["LapTime"]=track4['LapTime'].str.split('days ').str[1]
track4.loc[:, "LapTime (s)"] = track4["LapTime_TD"].dt.total_seconds()

In [6]:
qualy_laps="Japanese Grand Prix"
years_exclude=[2007,2008]
track_fastest=track4[(track4["GP"]==qualy_laps)&(~track4["Year"].isin(years_exclude))]

In [7]:
track_fastest["LapTime_TD"].min()

Timedelta('0 days 00:01:26.983000')

In [8]:
#Assign Rank for each entry point
track_fastest=track_fastest.dropna(subset=['LapTime'])
track_fastest["RK"] = track_fastest.groupby(by=["GP"])["LapTime (s)"].rank(method="dense", ascending=True)
track_fastest["RK"]= track_fastest["RK"].astype(int)
track_fastest.sort_values(by=["RK"]).head()

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,Position,ClassifiedPosition,GridPosition,Time,Status,Points,Year,GP,Type,LapTime,LapTime_TD,LapTime (s),RK
21249,1,M VERSTAPPEN,VER,,Red Bull Racing,3671C6,,Max,Verstappen,Max Verstappen,https://media.formula1.com/d_driver_fallback_i...,,1.0,,NaN,NaT,,NaN,2025,Japanese Grand Prix,Q3,00:01:26.983000,0 days 00:01:26.983000,86.983,1
21243,4,L NORRIS,NOR,,McLaren,FF8000,,Lando,Norris,Lando Norris,https://media.formula1.com/d_driver_fallback_i...,,2.0,,NaN,NaT,,NaN,2025,Japanese Grand Prix,Q3,00:01:26.995000,0 days 00:01:26.995000,86.995,2
21244,81,O PIASTRI,PIA,,McLaren,FF8000,,Oscar,Piastri,Oscar Piastri,https://media.formula1.com/d_driver_fallback_i...,,3.0,,NaN,NaT,,NaN,2025,Japanese Grand Prix,Q3,00:01:27.027000,0 days 00:01:27.027000,87.027,3
24445,5,S VETTEL,VET,VET,Ferrari,dc0000,None,Sebastian,Vettel,Sebastian Vettel,None,None,1.0,None,0.0,NaT,,0.0,2019,Japanese Grand Prix,Q3,00:01:27.064000,0 days 00:01:27.064000,87.064,4
10727,4,L NORRIS,NOR,,McLaren,FF8000,,Lando,Norris,Lando Norris,https://media.formula1.com/d_driver_fallback_i...,,2.0,,NaN,NaT,,NaN,2025,Japanese Grand Prix,Q2,00:01:27.146000,0 days 00:01:27.146000,87.146,5


In [9]:
pole_lap = track_fastest["LapTime_TD"].min()
track_fastest["Fastest_Lap"]=pole_lap
track_fastest['LapTimeDelta']= pd.to_timedelta(track_fastest["LapTime"]) - pd.to_timedelta(pole_lap)
track_fastest['LapTimeDelta']=track_fastest['LapTimeDelta'].astype(str)

for index, row in track_fastest.iterrows():
  if track_fastest.loc[index, 'LapTimeDelta']=='0 days 00:00:00':
    track_fastest.loc[index, 'LapTimeDelta']='0 days 00:00:00.01'
track_fastest['LapTimeDelta']=pd.to_timedelta(track_fastest['LapTimeDelta'])
track_fastest

track_fastest['LapTimeDelta'] = track_fastest['LapTimeDelta'] + pd.to_datetime('1970/01/01')

track_fastest['LapTimeDelta2']= pd.to_timedelta(track_fastest["LapTime"]) - pd.to_timedelta(pole_lap)
track_fastest['LapTimeDelta2']=track_fastest['LapTimeDelta2'].astype(str)
for index, row in track_fastest.iterrows():
  if track_fastest.loc[index, 'LapTimeDelta2']=='0 days 00:00:00':
    track_fastest.loc[index, 'LapTimeDelta2']='0 days 00:00:00.0001'

track_fastest["LapTimeDelta2"]=track_fastest['LapTimeDelta2'].str.split('days ').str[1]
track_fastest['LapTimeDelta2']=track_fastest['LapTimeDelta2'].str.split('00:').str[2]
track_fastest=track_fastest.sort_values(by=["LapTimeDelta2"])

track_fastest["Driver+Year"]=track_fastest["Abbreviation"]+" ("+track_fastest["Year"].astype(str)+" "+track_fastest["Type"]+")"

track_fastest.head()

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,Position,ClassifiedPosition,GridPosition,Time,Status,Points,Year,GP,Type,LapTime,LapTime_TD,LapTime (s),RK,Fastest_Lap,LapTimeDelta,LapTimeDelta2,Driver+Year
21249,1,M VERSTAPPEN,VER,,Red Bull Racing,3671C6,,Max,Verstappen,Max Verstappen,https://media.formula1.com/d_driver_fallback_i...,,1.0,,NaN,NaT,,NaN,2025,Japanese Grand Prix,Q3,00:01:26.983000,0 days 00:01:26.983000,86.983,1,0 days 00:01:26.983000,1970-01-01 00:00:00.010,00.0001,VER (2025 Q3)
21243,4,L NORRIS,NOR,,McLaren,FF8000,,Lando,Norris,Lando Norris,https://media.formula1.com/d_driver_fallback_i...,,2.0,,NaN,NaT,,NaN,2025,Japanese Grand Prix,Q3,00:01:26.995000,0 days 00:01:26.995000,86.995,2,0 days 00:01:26.983000,1970-01-01 00:00:00.012,00.012000,NOR (2025 Q3)
21244,81,O PIASTRI,PIA,,McLaren,FF8000,,Oscar,Piastri,Oscar Piastri,https://media.formula1.com/d_driver_fallback_i...,,3.0,,NaN,NaT,,NaN,2025,Japanese Grand Prix,Q3,00:01:27.027000,0 days 00:01:27.027000,87.027,3,0 days 00:01:26.983000,1970-01-01 00:00:00.044,00.044000,PIA (2025 Q3)
24445,5,S VETTEL,VET,VET,Ferrari,dc0000,None,Sebastian,Vettel,Sebastian Vettel,None,None,1.0,None,0.0,NaT,,0.0,2019,Japanese Grand Prix,Q3,00:01:27.064000,0 days 00:01:27.064000,87.064,4,0 days 00:01:26.983000,1970-01-01 00:00:00.081,00.081000,VET (2019 Q3)
10727,4,L NORRIS,NOR,,McLaren,FF8000,,Lando,Norris,Lando Norris,https://media.formula1.com/d_driver_fallback_i...,,2.0,,NaN,NaT,,NaN,2025,Japanese Grand Prix,Q2,00:01:27.146000,0 days 00:01:27.146000,87.146,5,0 days 00:01:26.983000,1970-01-01 00:00:00.163,00.163000,NOR (2025 Q2)


In [11]:
fig=px.bar(
    track_fastest.head(25),
    y="Driver+Year",
    x='LapTimeDelta',
    text='LapTimeDelta2',
    color="TeamName",
    orientation='h',
    template="presentation",
    hover_data=['TeamName', 'Year', 'GP','LapTime','RK'],
    color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "Racing Bulls": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "Alfa Romeo Racing":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
                 },
    title="<b>25 Fastest Qualifying Laps for the {}</b>".format(qualy_laps),
    height=800, 
    width=1200,
)
fig.update_layout(
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),

    xaxis_title="<b>Delta</b>",
    yaxis_title="<b>Driver</b>",
    title_font_family="<b>PT Sans Narrow</b>",

)

for x,y,z in zip(track_fastest.TeamName, track_fastest["Driver+Year"], track_fastest.LapTimeDelta):
  for png in (Path(Path.cwd()).parents[0].joinpath("F1_LOGOS").glob("*.png")):
    if str.split(str.split(str(png),".")[0],"/")[6]==x:
      image=str(png)
      fig.add_layout_image(
          x=z,
          y=y,
          source=Image.open(image),
          xref="x",
          yref="y",
          sizex=1,
          sizey=1,
          xanchor="center",
          yanchor="middle",
      )

fig.update_layout(xaxis_tickformat='%H:%M:%S.%f')

fig.update_traces(marker_line_width=1,marker_line_color="BLACK")

fig.update_layout(yaxis={'categoryorder':'total descending'})

fig.update_traces(textposition='auto')

fig.update_layout(
    title_x=0.5,
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=14,
        color="Black"
    ),
    title_font_family="PT Sans Narrow",
    margin=dict(l=110, r=10, t=35, b=60),
)
fig.show()